# TCP-paper sanity check (standalone)

Reproduces the comparison that justifies dropping TCP from the main grid:

* `garch_hardcoded` — TCP paper's exact GARCH (ω=1e-6, α=0.05, β=0.9, normal, no MLE).
* `garch_roll`     — properly fitted rolling GARCH(1,1) Student-t (parametric quantile).
* `fhs`            — Filtered Historical Simulation.
* `tcp_paper`      — paper-faithful TCP (gradient-boosted quantile + rolling split-conformal).

Headline result: on COVID 2020 individual equities, `garch_roll` and `fhs` both beat `tcp_paper` on Winkler score; `garch_hardcoded` undercoverere as expected.

In [ ]:
# sys.path bootstrap so 'from src...' works no matter where the kernel was launched from
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


In [ ]:
# TCP-paper sanity check
#
# Reproduces the garch_hardcoded vs garch_roll vs tcp_paper vs fhs
# comparison on the COVID-2020 crisis.  Documents WHY we dropped TCP
# from the main grid: with a properly fitted rolling GARCH (Student-t,
# parametric quantile) the TCP paper's apparent advantage disappears.
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from src.analysis._shared import ensure_data, iterate_experiments, walk_forward
from src.conformal.garch import GarchHardcoded, RollingGarch
from src.conformal.fhs import FHS
from src.conformal.tcp_paper import TCPPaperConformal
from src.metrics.metrics import Metrics
from src.metrics.prediction_cols import PredictionCols

CONFIDENCE_LEVEL = 0.90
NUM_LAGS = 5
REFIT_EVERY = 21

data = ensure_data()
TICKERS_2020 = {"MCD", "MGM", "AZO"}
splits_2020 = [s for s in iterate_experiments()
               if s.regime == "2020" and s.ticker.value in TICKERS_2020]

methods = [
    ("garch_hardcoded", lambda lc: GarchHardcoded(label_col_name=lc, confidence_level=CONFIDENCE_LEVEL)),
    ("garch_roll",      lambda lc: RollingGarch(label_col_name=lc, confidence_level=CONFIDENCE_LEVEL,
                                                 garch_window=504, refit_every=REFIT_EVERY)),
    ("fhs",             lambda lc: FHS(label_col_name=lc, confidence_level=CONFIDENCE_LEVEL)),
    ("tcp_paper",       lambda lc: TCPPaperConformal(label_col_name=lc, num_lags=NUM_LAGS,
                                                      confidence_level=CONFIDENCE_LEVEL,
                                                      refit_every=REFIT_EVERY)),
]

rows = []
for s in splits_2020:
    train = s.slice_train(data); calib = s.slice_calib(data); full = s.slice_full(data)
    for name, factory in methods:
        m = factory(s.label_col)
        if name == "fhs":
            m.fit_and_calibrate(pd.concat([train, calib]))
        else:
            m.fit_and_calibrate(train, calib)
        intervals = walk_forward(m, full, s.test_start, s.test_end)
        df = intervals.rename(columns={"r": PredictionCols.LOG_RETURNS,
                                        "l": PredictionCols.LOWER_BOUND,
                                        "u": PredictionCols.UPPER_BOUND})
        widths = df[PredictionCols.UPPER_BOUND] - df[PredictionCols.LOWER_BOUND]
        df = df[~np.isinf(widths)]
        met = Metrics(df, alpha=0.10)
        cov = met.get_coverage_metrics().get_marginal_coverage()
        wink, _, _ = met.get_winkler_metrics().get_mean_scores()
        width, _, _ = met.get_interval_metrics().get_mean_widths()
        rows.append({"method": name, "ticker": s.ticker.value,
                     "cov": cov, "winkler": wink, "width": width})

results = pd.DataFrame(rows)
display(results.round(3))

print("\nAverage across MCD, MGM, AZO  (lower Winkler = better)")
summary = (results.groupby("method")[["cov", "winkler", "width"]]
           .mean().sort_values("winkler").round(3))
display(summary)
